In [80]:
!pip install pyserini

In [81]:
import json
import os
from tqdm import tqdm

In [86]:
! ls /tmp/

In [84]:
collection_tsv_path = '/workspace/2404170001/inputs/collection.tsv' 
output_folder = '/tmp/pyserini_data'
os.makedirs(output_folder, exist_ok=True)
collection_jsonl_path = os.path.join(output_folder, 'collection.jsonl')


In [87]:
if not os.path.exists(collection_tsv_path):
    print(f"Error: The file '{collection_tsv_path}' does not exist. Please update the path.")
else:
    os.makedirs(output_folder, exist_ok=True)

    print(f"Converting {collection_tsv_path} to {collection_jsonl_path}...")
    
    with open(collection_tsv_path, 'r', encoding='utf-8') as in_file, \
         open(collection_jsonl_path, 'w', encoding='utf-8') as out_file:
        for line in tqdm(in_file):
            try:
                # Assumes the format is: passage_id\tpassage_text
                pid, passage_text = line.strip().split('\t')
                record = {"id": pid, "contents": passage_text}
                out_file.write(json.dumps(record) + '\n')
            except ValueError:
                print(f"Skipping malformed line: {line.strip()}")

    print("Conversion complete.")

In [89]:
!conda install -c conda-forge openjdk=21

In [91]:
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input /tmp/pyserini_data \
  --index /tmp/pyserini_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 8

In [93]:
!python -m pyserini.search.lucene \
      --index /tmp/pyserini_index \
      --topics /workspace/2404170001/inputs/queries.dev.small.tsv \
      --output /tmp/pyserini_data/bm25_run.txt \
      --bm25 \
      --hits 100 \
      --output-format trec 


In [94]:
def load_qrels(path):
    with open(path,'r') as f:
        qids_to_relevant_passageids = {}
        for l in f:
            try:
                l = l.strip().split()
                qid = l[0]
                if float(l[3]) > 0.0001:
                    if qid not in qids_to_relevant_passageids:
                        qids_to_relevant_passageids[qid] = {}
                    qids_to_relevant_passageids[qid][l[2]] = float(l[3])
            except:
                raise IOError('\"%s\" is not valid format' % l)
        return qids_to_relevant_passageids


In [106]:
qids_to_relevant_passageids

In [107]:
# // ...existing code...
#
# generate validation.tsv tuples from qrel files 
# -------------------------------
#

import os
import sys
from tqdm import tqdm
import random
random.seed(208973249)

#
# config
#
run_folder_2 = "/tmp/estop"
os.makedirs(run_folder_2, exist_ok=True)
val_out_file = os.path.join(run_folder_2, "validation.tsv")
candidate_file = '/workspace/2404170001/pyserini_data/bm25_run.txt'
qrel_file_path = '/workspace/2404170001/inputs/qrels.dev.tsv' 
collection_file_path = '/workspace/2404170001/inputs/collection.tsv'
query_file_path = '/workspace/2404170001/inputs/queries.dev.small.tsv'

# Parameters from the paper
max_doc_char_length = 100_000
number_of_sampled_queries = 3200
max_rank = 100

In [108]:
def load_qrels(path):
    with open(path,'r') as f:
        qids_to_relevant_passageids = {}
        for l in f:
            try:
                l = l.strip().split()
                qid = l[0]
                if float(l[3]) > 0: # A relevance score > 0
                    if qid not in qids_to_relevant_passageids:
                        qids_to_relevant_passageids[qid] = {}
                    qids_to_relevant_passageids[qid][l[2]] = float(l[3])
            except (IOError, IndexError, ValueError):
                print(f"Warning: Skipping malformed qrels line: {l}")
        return qids_to_relevant_passageids


In [109]:

print("Loading qrels...")
qrels = load_qrels(qrel_file_path)
qrels

In [110]:
print("Loading collection...")
collection = {}
with open(collection_file_path,"r",encoding="utf8") as collection_file_handle:
    for line in tqdm(collection_file_handle):
        ls = line.split("\t")
        if len(ls) == 2:
            _id, text = ls
            collection[_id] = text.rstrip()[:max_doc_char_length]


In [111]:
collection

In [112]:
print("Loading queries...")
queries = {}
with open(query_file_path,"r",encoding="utf8") as query_file_handle:
    for line in tqdm(query_file_handle):
        ls = line.split("\t")
        if len(ls) == 2:
            _id, text = ls
            queries[_id] = text.rstrip()

In [113]:
queries

In [115]:
len(list(qrels.keys() & queries.keys()))

In [116]:

# Get all unique query IDs from the qrels file that are also in our queries file
all_query_ids = list(qrels.keys() & queries.keys())

# Uniformly sample 3,200 queries as described in the paper
if len(all_query_ids) < number_of_sampled_queries:
    print(f"Warning: Only {len(all_query_ids)} queries available, sampling all of them.")
    sampled_query_ids = set(all_query_ids)
else:
    sampled_query_ids = set(random.sample(all_query_ids, number_of_sampled_queries))

print(f"Sampled {len(sampled_query_ids)} queries for the validation set.")

In [118]:

known_pairs = set()

with open(val_out_file,"w",encoding="utf8") as val_out_file_handle:
    # First, add the top 100 candidates from the baseline run file
    print("Processing candidate file...")
    with open(candidate_file,"r",encoding="utf8") as candidate_file_handle:
        for line in tqdm(candidate_file_handle):
            ls = line.split() # TREC format: qid Q0 pid rank score run_name
            print(ls)
            if len(ls) < 4: continue

            query_id = ls[0]
            doc_id = ls[2]   # Corrected: pid is the 3rd element
            rank = int(ls[3]) # Corrected: rank is the 4th element

            if query_id not in sampled_query_ids:
                continue

            if rank > max_rank:
                continue

            if (query_id, doc_id) not in known_pairs:
                if query_id in queries and doc_id in collection:
                    known_pairs.add((query_id, doc_id))
                    out_arr = [query_id, doc_id, queries[query_id], collection[doc_id]]
                    val_out_file_handle.write("\t".join(out_arr) + "\n")
                    
     # Second, add all relevant passages that might have been missed by the baseline
    print("Adding missing relevant passages from qrels...")
    for q_id in tqdm(sampled_query_ids):
        if q_id in qrels:
            for rel_doc in qrels[q_id]:
                if (q_id, rel_doc) not in known_pairs:
                    if q_id in queries and rel_doc in collection:
                        known_pairs.add((q_id, rel_doc))
                        out_arr = [q_id, rel_doc, queries[q_id], collection[rel_doc]]
                        val_out_file_handle.write("\t".join(out_arr) + "\n")


In [101]:
# Add this validation cell after your generation code

import pandas as pd

# Load and inspect the generated file
validation_file = "/tmp/estop/validation.tsv"
print("=== Basic File Validation ===")

# Check file exists and size
import os
if os.path.exists(validation_file):
    file_size = os.path.getsize(validation_file)
    print(f"✓ File exists: {validation_file}")
    print(f"✓ File size: {file_size:,} bytes")
else:
    print("✗ File does not exist!")

# Read and inspect structure
try:
    df = pd.read_csv(validation_file, sep='\t', header=None, 
                     names=['query_id', 'doc_id', 'query_text', 'doc_text'])
    print(f"✓ Total pairs: {len(df):,}")
    print(f"✓ Unique queries: {df['query_id'].nunique():,}")
    print(f"✓ Unique documents: {df['doc_id'].nunique():,}")
    print(f"✓ Avg pairs per query: {len(df) / df['query_id'].nunique():.1f}")
    print("\n=== Sample rows ===")
    print(df.head(3))
except Exception as e:
    print(f"✗ Error reading file: {e}")

In [102]:
print("\n=== Paper Compliance Check ===")

# Check query sampling (should be ~3,200 queries)
expected_queries = 3200
actual_queries = df['query_id'].nunique()
print(f"Expected queries: {expected_queries}")
print(f"Actual queries: {actual_queries}")
if abs(actual_queries - expected_queries) <= 100:  # Allow some tolerance
    print("✓ Query count matches paper specification")
else:
    print("⚠ Query count differs from paper specification")

# Check that we have both BM25 candidates AND relevant passages
bm25_pairs = set()
relevant_pairs = set()

# Load BM25 run file to see what should be there
with open('/workspace/2404170001/pyserini_data/bm25_run.txt', 'r') as f:
    for line in f:
        ls = line.split()
        if len(ls) >= 4:
            qid, doc_id, rank = ls[0], ls[2], int(ls[3])
            if qid in sampled_query_ids and rank <= 100:
                bm25_pairs.add((qid, doc_id))

# Check qrels for relevant pairs
for qid in sampled_query_ids:
    if qid in qrels:
        for doc_id in qrels[qid]:
            relevant_pairs.add((qid, doc_id))

print(f"\nBM25 candidate pairs: {len(bm25_pairs):,}")
print(f"Relevant pairs from qrels: {len(relevant_pairs):,}")
print(f"Overlap between BM25 and relevant: {len(bm25_pairs & relevant_pairs):,}")
print(f"Total unique pairs expected: {len(bm25_pairs | relevant_pairs):,}")
print(f"Actual pairs in file: {len(df):,}")

if len(df) >= len(bm25_pairs | relevant_pairs) * 0.95:  # Allow 5% tolerance
    print("✓ Pair count looks reasonable")
else:
    print("⚠ Missing pairs detected")

In [103]:
print("\n=== Content Quality Check ===")

# Check for missing or malformed data
null_count = df.isnull().sum().sum()
empty_count = (df == '').sum().sum()
print(f"Null values: {null_count}")
print(f"Empty values: {empty_count}")

# Check text lengths (should be reasonable)
avg_query_len = df['query_text'].str.len().mean()
avg_doc_len = df['doc_text'].str.len().mean()
print(f"Average query length: {avg_query_len:.0f} chars")
print(f"Average document length: {avg_doc_len:.0f} chars")

# Check for duplicates
duplicates = df.duplicated(subset=['query_id', 'doc_id']).sum()
print(f"Duplicate query-doc pairs: {duplicates}")

# Sample a few specific pairs to manually verify
print("\n=== Manual Spot Check ===")
sample_pairs = df.sample(3)
for _, row in sample_pairs.iterrows():
    qid, did = row['query_id'], row['doc_id']
    print(f"\nQuery {qid}: {row['query_text'][:100]}...")
    print(f"Doc {did}: {row['doc_text'][:100]}...")
    
    # Check if this pair exists in our source data
    in_queries = qid in queries
    in_collection = did in collection
    in_bm25 = (qid, did) in bm25_pairs
    in_qrels = qid in qrels and did in qrels.get(qid, {})
    
    print(f"  ✓ Query in source: {in_queries}")
    print(f"  ✓ Doc in collection: {in_collection}")
    print(f"  ✓ From BM25: {in_bm25}")
    print(f"  ✓ From qrels: {in_qrels}")

In [104]:
print("\n=== Cross-Reference Validation ===")

# Verify all query IDs in the file exist in sampled set
file_query_ids = set(df['query_id'].unique())
missing_from_sample = file_query_ids - sampled_query_ids
extra_in_file = sampled_query_ids - file_query_ids

print(f"Queries in file but not in sample: {len(missing_from_sample)}")
print(f"Queries in sample but not in file: {len(extra_in_file)}")

if missing_from_sample:
    print(f"  Unexpected queries: {list(missing_from_sample)[:5]}...")
if extra_in_file:
    print(f"  Missing queries: {list(extra_in_file)[:5]}...")

# Check that text matches between file and source
print("\nText consistency check...")
mismatches = 0
for i, row in df.head(10).iterrows():  # Check first 10 rows
    qid, did = row['query_id'], row['doc_id']
    if qid in queries and did in collection:
        if row['query_text'] != queries[qid]:
            print(f"Query text mismatch for {qid}")
            mismatches += 1
        if row['doc_text'] != collection[did]:
            print(f"Doc text mismatch for {did}")
            mismatches += 1

print(f"Text mismatches in sample: {mismatches}")
if mismatches == 0:
    print("✓ Text consistency check passed")

In [105]:
print("\n" + "="*50)
print("VALIDATION SUMMARY")
print("="*50)

# Calculate key metrics
total_pairs = len(df)
unique_queries = df['query_id'].nunique()
unique_docs = df['doc_id'].nunique()
avg_pairs_per_query = total_pairs / unique_queries

validation_checks = [
    ("File exists and readable", os.path.exists(validation_file)),
    ("Query count ~3200", abs(unique_queries - 3200) <= 100),
    ("No null/empty values", null_count + empty_count == 0),
    ("No duplicate pairs", duplicates == 0),
    ("Text consistency", mismatches == 0),
    ("Reasonable pair count", total_pairs > unique_queries * 50),  # At least 50 pairs per query on average
]

print(f"Total pairs: {total_pairs:,}")
print(f"Unique queries: {unique_queries:,}")
print(f"Unique documents: {unique_docs:,}")
print(f"Avg pairs per query: {avg_pairs_per_query:.1f}")
print()

for check_name, passed in validation_checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}: {check_name}")

all_passed = all(check[1] for check in validation_checks)
print(f"\nOVERALL: {'✓ VALIDATION SUCCESSFUL' if all_passed else '✗ VALIDATION FAILED'}")